In [398]:
import pandas as pd
import joblib
import numpy as np
import json

In [399]:
# Risk models
lc_model = joblib.load("../../models/lc_calibrated_gradient_boosting_model.pkl")
lt_model = joblib.load("../../models/lt_calibrated_gradient_boosting_model.pkl")

# APR models
apr_mean = joblib.load("../../models/apr_model_mean.pkl")
apr_low = joblib.load("../../models/apr_model_low.pkl")
apr_high = joblib.load("../../models/apr_model_high.pkl")

# Features
with open("../../models/apr_features.json") as f:
    apr_features = json.load(f)

In [400]:
df_lc = pd.read_csv('../../data/processed/featured_lending-club-car-loan.csv')
df_lt = pd.read_csv('../../data/processed/featured_lt-vehicle-loan_train.csv')

In [401]:
df = df_lc.copy()

In [402]:
df = pd.get_dummies(df, columns=["verification_status"], drop_first=False)

In [403]:
df["term_x_score"] = df["term"] * df["fico_score"]
df["loan_x_score"] = df["log_loan_amount"] * df["fico_score"]

In [404]:
model_lc = lc_model["model"]
lc_features = lc_model["features"]
lc_model["threshold"] = 0.35

In [405]:
lt_features = [
    "log_loan_amount",
    "log_installment",
    "log_delinquency",
    "log_inquiries",
    "log_credit_history_length",
    "credit_score_norm",
    "ltv_norm"
]

In [406]:
# LC risk
df["p_lc"] = model_lc.predict_proba(df[lc_features])[:, 1]

In [407]:
# L&T risk as proxy
df["p_lt"] = df["p_lc"] * 0.9 + 0.02

In [408]:
df["apr_mean"] = apr_mean.predict(df[apr_features])
df["apr_low"] = apr_low.predict(df[apr_features])
df["apr_high"] = apr_high.predict(df[apr_features])

# Clip (safety)
df["apr_mean"] = df["apr_mean"].clip(5, 35)
df["apr_low"] = df["apr_low"].clip(5, 35)
df["apr_high"] = df["apr_high"].clip(5, 35)

In [409]:
df["loan_amount"] = np.expm1(df["log_loan_amount"])

In [410]:
# Financial Assumptions
LGD_LC = 0.9
LGD_LT = 0.5

In [411]:
# Mean scenario
df["profit_lc_mean"] = (
    df["apr_mean"] * df["loan_amount"]
    - df["p_lc"] * LGD_LC * df["loan_amount"]
)

df["profit_lt_mean"] = (
    df["apr_mean"] * df["loan_amount"]
    - df["p_lt"] * LGD_LT * df["loan_amount"]
)

In [412]:
# Conservative (lower APR)
df["profit_lc_low"] = (
    df["apr_low"] * df["loan_amount"]
    - df["p_lc"] * LGD_LC * df["loan_amount"]
)

df["profit_lt_low"] = (
    df["apr_low"] * df["loan_amount"]
    - df["p_lt"] * LGD_LT * df["loan_amount"]
)

In [413]:
# Aggressive (higher APR)
df["profit_lc_high"] = (
    df["apr_high"] * df["loan_amount"]
    - df["p_lc"] * LGD_LC * df["loan_amount"]
)

df["profit_lt_high"] = (
    df["apr_high"] * df["loan_amount"]
    - df["p_lt"] * LGD_LT * df["loan_amount"]
)

Risk Segmentation

In [414]:
threshold = lc_model["threshold"]

def get_risk_segment(p):
    if p < 0.12:
        return "Prime"
    elif p < 0.25:
        return "Near-Prime"
    elif p < threshold:
        return "Subprime"
    else:
        return "Reject"

df["segment"] = df["p_lc"].apply(get_risk_segment)

Decision (Accept / Reject)

In [415]:
df["decision"] = df["segment"].apply(
    lambda x: "Reject" if x == "Reject" else "Approve"
)

Lender Assignment

In [416]:
def assign_lender(row):

    if row["decision"] == "Reject":
        return None

    if row["segment"] == "Prime":
        return "LendingClub"

    if row["segment"] == "Subprime":
        return "L&T"

    # Near-Prime → profit-based
    return "LendingClub" if row["profit_lc_mean"] > row["profit_lt_mean"] else "L&T"

df["lender"] = df.apply(assign_lender, axis=1)

Profit

In [417]:
def final_profit(row):
    if row["decision"] == "Reject":
        return 0
    return row["profit_lc_mean"] if row["lender"] == "LendingClub" else row["profit_lt_mean"]

df["final_profit"] = df.apply(final_profit, axis=1)

System Evaluation

In [418]:
# Approval Rate
approval_rate = (df["decision"] == "Approve").mean()
print("Approval Rate:", approval_rate)

# Portfolio distribution across segments
approved_df = df[df["decision"] == "Approve"]
segment_distribution = approved_df["segment"].value_counts(normalize=True)
full_distribution = df["segment"].value_counts(normalize=True)
print("Loan Distribution by Segment:\n", full_distribution)


# Avg Profit by Segment
profit_by_segment = df.groupby("segment")["final_profit"].mean()
print("\nAvg Profit by Segment:\n", profit_by_segment)

# Avg Risk by Segment
risk_by_segment = df.groupby("segment")["p_lc"].mean()
print("\nAvg Risk by Segment:\n", risk_by_segment)

Approval Rate: 0.868950890711958
Loan Distribution by Segment:
 segment
Near-Prime    0.500719
Prime         0.195296
Subprime      0.172935
Reject        0.131049
Name: proportion, dtype: float64

Avg Profit by Segment:
 segment
Near-Prime    168282.353950
Prime         141983.671815
Reject             0.000000
Subprime      228064.971815
Name: final_profit, dtype: float64

Avg Risk by Segment:
 segment
Near-Prime    0.180585
Prime         0.085801
Reject        0.436171
Subprime      0.292157
Name: p_lc, dtype: float64


Pricing Strategy

In [419]:
def select_apr(row):

    if row["segment"] == "Prime":
        return row["apr_low"]

    elif row["segment"] == "Near-Prime":
        return row["apr_mean"]

    elif row["segment"] == "Subprime":
        return row["apr_high"]

    return None

df["final_apr"] = df.apply(select_apr, axis=1)

# Policy Tuning

In [420]:
thresholds = [0.30, 0.32, 0.35, 0.38, 0.40]

In [421]:
def evaluate_policy(df, threshold):

    # --- Segment ---
    def segment(p):
        if p < 0.12:
            return "Prime"
        elif p < 0.25:
            return "Near-Prime"
        elif p < threshold:
            return "Subprime"
        else:
            return "Reject"

    df_temp = df.copy()
    df_temp["segment"] = df_temp["p_lc"].apply(segment)

    df_temp["decision"] = df_temp["segment"].apply(
        lambda x: "Reject" if x == "Reject" else "Approve"
    )

    # --- Approved only ---
    approved = df_temp[df_temp["decision"] == "Approve"]

    approval_rate = len(approved) / len(df_temp)

    avg_risk = approved["p_lc"].mean()

    avg_profit = approved["final_profit"].mean()

    segment_dist = approved["segment"].value_counts(normalize=True)

    return {
        "threshold": threshold,
        "approval_rate": approval_rate,
        "avg_risk": avg_risk,
        "avg_profit": avg_profit,
        "segment_dist": segment_dist
    }

In [422]:
results = []

for t in thresholds:
    res = evaluate_policy(df, t)
    results.append(res)

for r in results:
    print("\nThreshold:", r["threshold"])
    print("Approval:", round(r["approval_rate"], 3))
    print("Risk:", round(r["avg_risk"], 3))
    print("Profit:", round(r["avg_profit"], 0))
    print("Segment:", r["segment_dist"].to_dict())


Threshold: 0.3
Approval: 0.803
Risk: 0.17
Profit: 167156.0
Segment: {'Near-Prime': 0.6234477081967896, 'Prime': 0.2431642050280702, 'Subprime': 0.13338808677514014}

Threshold: 0.32
Approval: 0.833
Risk: 0.175
Profit: 169939.0
Segment: {'Near-Prime': 0.6011825360888545, 'Prime': 0.23448008797341216, 'Subprime': 0.1643373759377333}

Threshold: 0.35
Approval: 0.869
Risk: 0.181
Profit: 174269.0
Segment: {'Near-Prime': 0.57623432042101, 'Prime': 0.22474949958567622, 'Subprime': 0.19901617999331384}

Threshold: 0.38
Approval: 0.899
Risk: 0.188
Profit: 168503.0
Segment: {'Near-Prime': 0.5571680062348453, 'Subprime': 0.22551895608468567, 'Prime': 0.21731303768046903}

Threshold: 0.4
Approval: 0.916
Risk: 0.191
Profit: 165276.0
Segment: {'Near-Prime': 0.546496659398501, 'Subprime': 0.24035246365387428, 'Prime': 0.21315087694762472}


# Finding:
threshold = 0.35
Maximum profit: 174269.0

In [423]:
approved = df[df["decision"] == "Approve"]

approval_rate = len(approved) / len(df)
avg_risk = approved["p_lc"].mean()
avg_profit = approved["final_profit"].mean()

segment_mix = approved["segment"].value_counts(normalize=True)

print("Approval Rate:", approval_rate)
print("Avg Risk:", avg_risk)
print("Avg Profit:", avg_profit)
print("Segment Mix:\n", segment_mix)

Approval Rate: 0.868950890711958
Avg Risk: 0.18148675266022968
Avg Profit: 174269.44653815863
Segment Mix:
 segment
Near-Prime    0.576234
Prime         0.224749
Subprime      0.199016
Name: proportion, dtype: float64


# Routing Function

In [424]:
def apply_hard_rules(input_dict):

    income = input_dict.get("annual_inc", 0)
    installment = input_dict.get("installment", 0)
    loan = input_dict.get("loan_amnt", 0)
    dti = input_dict.get("dti", 0)
    fico = input_dict.get("fico_score", 0)

    # --- Rule 1: Installment affordability ---
    if income > 0 and installment / income > 0.5:
        return "Installment too high vs income"

    # --- Rule 2: Loan too large vs income ---
    if income > 0 and loan / income > 10:
        return "Loan too large relative to income"

    # --- Rule 3: DTI cap ---
    if dti > 0.6:
        return "DTI too high"

    # --- Rule 4: Very low income ---
    if income < 5000:
        return "Income too low"

    # --- Rule 5: Very poor credit ---
    if fico < 580:
        return "Credit score too low"

    return None

In [425]:
def prepare_features(input_dict):
    import pandas as pd
    import numpy as np

    df = pd.DataFrame([input_dict])

    # --- Log transforms ---
    df["log_loan_amount"] = np.log1p(df["loan_amnt"])
    df["log_installment"] = np.log1p(df["installment"])
    df["log_income"] = np.log1p(df["annual_inc"])
    df["log_dti"] = np.log1p(df["dti"])

    # Optional (if present)
    df["log_inquiries"] = np.log1p(df.get("inquiries", 0))
    df["log_delinquency"] = np.log1p(df.get("delinq_2yrs", 0))

    # Add missing features with defaults
    df["total_acc"] = 10
    df["emp_length"] = 2
    df["verification_status_Source Verified"] = 0

    # --- Interaction features ---
    df["term_x_score"] = df["term"] * df["fico_score"]
    df["loan_x_score"] = df["log_loan_amount"] * df["fico_score"]

    # --- One-hot encoding ---
    if "verification_status" in df.columns:
        df = pd.get_dummies(df, columns=["verification_status"], drop_first=False)

    return df

In [426]:
def predict_risk(df_input, lc_model):

    lc_features = lc_model["features"]

    for col in lc_features:
        if col not in df_input.columns:
            df_input[col] = 0

    df_lc = df_input[lc_features]

    p_lc = lc_model["model"].predict_proba(df_lc)[0, 1]
    p_lt = p_lc * 0.9 + 0.02

    return p_lc, p_lt

In [427]:
def get_segment(p_lc, threshold):

    if p_lc < 0.12:
        return "Prime"
    elif p_lc < 0.25:
        return "Near-Prime"
    elif p_lc < threshold:
        return "Subprime"
    else:
        return "Reject"

In [428]:
def predict_apr(df_input, apr_models, apr_features):
    import numpy as np

    for col in apr_features:
        if col not in df_input.columns:
            df_input[col] = 0

    df_apr = df_input[apr_features]

    apr_mean, apr_low, apr_high = apr_models

    apr_m = np.clip(apr_mean.predict(df_apr)[0], 5, 35)
    apr_l = np.clip(apr_low.predict(df_apr)[0], 5, 35)
    apr_h = np.clip(apr_high.predict(df_apr)[0], 5, 35)

    return apr_m, apr_l, apr_h

In [429]:
def select_apr(segment, apr_m, apr_l, apr_h):

    if segment == "Prime":
        return apr_l
    elif segment == "Near-Prime":
        return apr_m
    else:
        return apr_h

In [430]:
def compute_profit(loan, apr, term, p_lc, p_lt):

    LGD_LC = 0.9
    LGD_LT = 0.5

    # Convert term to years
    t = term / 12

    revenue = apr * loan * t

    loss_lc = p_lc * LGD_LC * loan
    loss_lt = p_lt * LGD_LT * loan

    return revenue - loss_lc, revenue - loss_lt

In [431]:
def assign_lender(segment, profit_lc, profit_lt):

    if segment == "Prime":
        return "LendingClub"

    elif segment == "Subprime":
        return "L&T"

    else:
        return "LendingClub" if profit_lc > profit_lt else "L&T"

In [434]:
def route_loan(input_dict, lc_model, apr_models, apr_features):

    # --- Step 1: Hard rules ---
    rule = apply_hard_rules(input_dict)
    if rule:
        return {
            "decision": "Reject",
            "reason": rule
        }

    # --- Step 2: Feature prep ---
    df_input = prepare_features(input_dict)

    # --- Step 3: Risk ---
    p_lc, p_lt = predict_risk(df_input, lc_model)

    threshold = lc_model["threshold"]

    # --- Step 4: Segment ---
    segment = get_segment(p_lc, threshold)

    if segment == "Reject":
        return {
            "decision": "Reject",
            "segment": "Reject",
            "p_lc": round(p_lc, 4)
        }

    # --- Step 5: APR ---
    apr_m, apr_l, apr_h = predict_apr(df_input, apr_models, apr_features)

    apr = select_apr(segment, apr_m, apr_l, apr_h)

    # --- Step 6: Profit ---
    loan = input_dict["loan_amnt"]
    term = input_dict["term"]

    profit_lc, profit_lt = compute_profit(loan, apr/100, term, p_lc, p_lt)

    # --- Step 7: Lender ---
    lender = assign_lender(segment, profit_lc, profit_lt)

    return {
        "decision": "Approve",
        "segment": segment,
        "lender": lender,
        "p_lc": round(p_lc, 4),
        "p_lt": round(p_lt, 4),
        "apr": round(apr, 2),
        "profit": round(max(profit_lc, profit_lt), 2)
    }

Example Usage

In [438]:
input_data = {
    "loan_amnt": 500_000,
    "installment": 15_000,
    "annual_inc": 80_000,
    "term": 36,
    "fico_score": 720,
    "dti": 0.4,
    "revol_util": 0.45,
}

result = route_loan(input_data, lc_model, (apr_mean, apr_low, apr_high), apr_features)

print(f"Decision: {result["decision"]}")
if result["decision"] == "Approve":
    print(f"Segment: {result["segment"]}")
    print(f"Lender: {result["lender"]}")
    print(f"Probability of default using your LendingClub-trained risk model: {result["p_lc"]}")
    print(f"Probability of default using your L&T(secured loan) risk model: {result["p_lt"]}")
    print(f"APR: {result["apr"]}")
    print(f"Expected Profit: {result["profit"]}")

Decision: Approve
Segment: Prime
Lender: LendingClub
Probability of default using your LendingClub-trained risk model: 0.1181
Probability of default using your L&T(secured loan) risk model: 0.1263
APR: 8.13
Expected Profit: 90363.38
